In [1]:
# Load all artifacts needed for hyperparameter optimisation.

import sys
sys.path.insert(0, '..')

import os

import time
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer

from src.estimators      import XGBoostDst, LightGBMDst
from src.mlflow_tracking import setup_mlflow, safe_mlflow_run
from src.runner   import fit_model, run_segment
from src.evaluate import compute_metrics, neg_storm_rmse
from src.splits import BOUNDARIES, PURGE_H, build_masks
from src.features import build_feature_sets
from src.config import STORM_THR, K_HORIZONS, EXPERIMENT_NAME



In [2]:
# H_STAR: selected forecasting horizon in hours.
# DST_TARGET_H: name of the target column corresponding to that horizon.

# H_STAR is defined locally in this notebook rather than in src/config.py, since the
# forecasting horizon is a research decision under active revision, not a fixed project-wide
# constant. DST_TARGET_H is derived from H_STAR so that every downstream reference to the
# target column stays in sync with the chosen horizon automatically, avoiding the kind of
# hardcoded-value drift that previously caused this notebook to train on h=7 while the rest
# of the project had already moved to h=3.
H_STAR = 7
DST_TARGET_H = f'dst_target_{H_STAR}h'

# Geomagnetic Storm Prediction from Cosmic Ray Measurements — Reference Model Performance & Hyperparameter Optimisation

This notebook performs reference model evaluation and hyperparameter optimisation for XGBoost and LightGBM at $h^* = 7$h with the MODEL_D feature set (OMNI + neutron features). Results are saved to `models/hp_opt_results.pkl` for use in the main ML notebook.

Loads the three artifacts produced by earlier stages: the feature matrix (`feat_split.parquet`), shared project constants (`context_constants.pkl`), and the selected predictor subset with its scaler (`feature_selection_results.pkl`).

In [3]:
feat_data  = pd.read_parquet('../data/processed/feat_split.parquet')
ctx_data   = joblib.load('../models/context_constants.pkl')
fs_data    = joblib.load('../models/feature_selection_results.pkl')

setup_mlflow()


## Reference Model Performance

Both XGBoost and LightGBM are first evaluated with the reference hyperparameter configuration on Train_1 + Train_2. This establishes a pre-tuning baseline against which the optimised models are compared. Since the forecasting horizon, feature set and training data remain fixed throughout, any performance differences after hyperparameter optimisation can be attributed to the optimisation process itself.

### Feature Sets

The predictor configuration is fixed from the scientific validation — MODEL_D. Feature sets are constructed from the selected features using `src/features.py`.

In [4]:
feature_sets          = build_feature_sets(fs_data['selected_final'])
MODEL_D_OMNI_HISTORY  = feature_sets['MODEL_D_OMNI_HISTORY']
OMNI_FEATURES         = feature_sets['OMNI_FEATURES']

print(f'MODEL_D features: {len(MODEL_D_OMNI_HISTORY)}')
print(MODEL_D_OMNI_HISTORY)

MODEL_D features: 23
['bz_gsm', 'sw_speed', 'sw_density', 'sw_pressure', 'e_field', 'mach_alfven', 'f107', 'ssn', 'bz_acc_3h', 'bz_acc_6h', 'bz_acc_12h', 'bz_gsm_lag3', 'bz_gsm_lag7', 'bz_gsm_lag21', 'sw_speed_lag1', 'sw_speed_lag3', 'sw_speed_lag7', 'sw_speed_lag21', 'solar_sin', 'solar_cos', 'd_neutron', 'neutron_counts_lag12', 'neutron_counts_lag3']


### Data Splits

| Segment | Period | Role |
|---|---|---|
| Train_1 | 1995–2003-10-14 | Training |
| Val_Storm | 2003-10-15 – 2003-12-15 | External validation — extreme storm (excluded from training) |
| Train_2 | 2003-12-16 – 2008-12-31 | Training |
| Val_Main | 2009–2014 | External validation — routine conditions |

Train_1 + Train_2 combined (`train_mask`) is used for training and hyperparameter search. Val_Storm is excluded from `train_mask` despite falling within the 1995–2008 date range. Val_Main and Val_Storm are used only for post-optimisation evaluation — never during training or the hyperparameter search itself.

`y_train` (used only as the MASE denominator) is drawn from `train1_mask` rather than the combined `train_mask`, and rather than the segment being evaluated (`val_main`/`val_storm`). The MASE denominator [HYN06] is defined as an in-sample, one-step scale computed on the training series — using the evaluation segment itself would make the metric self-referential, and using `train_mask` would introduce an artificial jump at the Train_1/Train_2 boundary where Val_Storm is excluded. `train1` is chronologically continuous and is applied as a single fixed scale across both `val_main` and `val_storm`, keeping their MASE values comparable to each other.

Builds the segment masks via `build_masks()` and reports row counts for `Train₁+₂`, `Val_Main`, `Val_Storm`, and confirms no missing values remain in the training target.

In [5]:
# ── Splits & Training data ────────────────────────────────────────────────
# build_masks() constructs all segment masks from src/splits.py
# train1_mask is needed only for the MASE denominator
masks          = build_masks(feat_data['datetime'])
train_mask     = masks['train']        # Train_1 + Train_2
train1_mask    =  masks['train1']
val_main_mask  = masks['val_main']
val_storm_mask = masks['val_storm']

EVAL_SEGMENTS = {
    'val_main' : val_main_mask,
    'val_storm': val_storm_mask,
}

X_train_full = feat_data.loc[train_mask, MODEL_D_OMNI_HISTORY]
y_train_full = feat_data.loc[train_mask, DST_TARGET_H]
y_train      = feat_data.loc[train1_mask, 'dst'].copy()  # MASE denominator

print(f'Train_1+2 rows     : {train_mask.sum():,}')
print(f'Val_main rows      : {val_main_mask.sum():,}')
print(f'Val_storm rows     : {val_storm_mask.sum():,}')
print(f'NaN in y_train_full: {y_train_full.isna().sum()}')

Train_1+2 rows     : 121,185
Val_main rows      : 52,542
Val_storm rows     : 1,446
NaN in y_train_full: 0


### Candidate Model Training

Both models are trained on Train_1 + Train_2 (1995–2008) with the reference hyperparameter configuration below. These results serve as the pre-tuning baseline.

| Parameter | XGBoost | LightGBM |
|---|---|---|
| `n_estimators` | 500 | 500 |
| `max_depth` | 5 | 5 |
| `learning_rate` | 0.05 | 0.05 |
| `subsample` | 0.8 | 0.8 |
| `colsample_bytree` | 0.8 | 0.8 |
| `num_leaves` | — | 31 |

In [6]:
# ── Candidate Model Training ──────────────────────────────────────────────
# Reference hyperparameters — pre-tuning baseline
xgb_pipe  = Pipeline([('model', XGBoostDst())])
xgb_pipe.fit(X_train_full, y_train_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['bz_gsm', 'sw...ed', 'sw...ty', 'sw...re', ...]"
,n_estimators,500
,max_depth,5
,learning_rate,0.05
,subsample,0.8
,colsample_bytree,0.8


In [7]:
lgbm_pipe = Pipeline([('model', LightGBMDst())])
lgbm_pipe.fit(X_train_full, y_train_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['bz_gsm', 'sw...ed', 'sw...ty', 'sw...re', ...]"
,n_estimators,500
,max_depth,5
,learning_rate,0.05
,num_leaves,31
,subsample,0.8


### Reference Performance — Results

Both reference models are evaluated on Val_Main and Val_Storm before hyperparameter optimisation. These results establish the pre-tuning baseline against which the optimised models will be compared.

In [8]:
# ── Reference Performance ─────────────────────────────────────────────────
print('\n── Reference performance (before tuning) ────────────────────────')
print(f'{"Model":<12} {"Segment":<12} {"RMSE":>8} {"StormRMSE":>12} {"MASE":>8}')
print('─' * 55)

for model_name, pipe in [('XGBoost', xgb_pipe), ('LightGBM', lgbm_pipe)]:
    for seg_name, seg_mask in EVAL_SEGMENTS.items():
        m = compute_metrics(
            y_true    = feat_data.loc[seg_mask, DST_TARGET_H],
            y_pred    = pipe.predict(feat_data.loc[seg_mask, MODEL_D_OMNI_HISTORY]),
            y_train   = y_train,
            y_persist = feat_data.loc[seg_mask, 'dst'].values,
            storm_thr = STORM_THR,
            horizon   = H_STAR,
        )
        print(f'{model_name:<12} {seg_name:<12} '
              f'{m["rmse"]:>8.2f} {m["storm_rmse"]:>12.2f} {m["mase"]:>8.3f}')


── Reference performance (before tuning) ────────────────────────
Model        Segment          RMSE    StormRMSE     MASE
───────────────────────────────────────────────────────
XGBoost      val_main        13.54        29.41    3.292
XGBoost      val_storm       39.38       102.19    5.583
LightGBM     val_main        13.65        28.75    3.315
LightGBM     val_storm       40.06       103.96    5.671


Diagnostic cell — reports the current kernel's `H_STAR` value, `MODEL_D_OMNI_HISTORY` feature count and contents, and confirms the target column and training mask sizes match expectations before proceeding to hyperparameter search.

In [9]:
# ── Diagnostic: sanity-check current kernel state ──────────────────────────
print(f"H_STAR value        : {H_STAR}")
print(f"MODEL_D feature count: {len(MODEL_D_OMNI_HISTORY)}")
print(f"MODEL_D features     : {MODEL_D_OMNI_HISTORY}")
print(f"'{DST_TARGET_H}' in feat_data.columns: {DST_TARGET_H in feat_data.columns}")
print(f"train_mask.sum()     : {train_mask.sum():,}")
print(f"y_train_full shape   : {y_train_full.shape}")
print(f"y_train_full.name    : {y_train_full.name}")

H_STAR value        : 7
MODEL_D feature count: 23
MODEL_D features     : ['bz_gsm', 'sw_speed', 'sw_density', 'sw_pressure', 'e_field', 'mach_alfven', 'f107', 'ssn', 'bz_acc_3h', 'bz_acc_6h', 'bz_acc_12h', 'bz_gsm_lag3', 'bz_gsm_lag7', 'bz_gsm_lag21', 'sw_speed_lag1', 'sw_speed_lag3', 'sw_speed_lag7', 'sw_speed_lag21', 'solar_sin', 'solar_cos', 'd_neutron', 'neutron_counts_lag12', 'neutron_counts_lag3']
'dst_target_7h' in feat_data.columns: True
train_mask.sum()     : 121,185
y_train_full shape   : (121185,)
y_train_full.name    : dst_target_7h


## Hyperparameter Optimisation

Hyperparameter optimisation targets storm-period performance directly via `neg_storm_rmse` as the CV scorer.

**Search protocol:** RandomizedSearchCV (n_iter=50, random_state=42) with TimeSeriesSplit (5 folds, gap=H_STAR) on Train_1 + Train_2. Val_Main and Val_Storm are not used during the search.

**Produces:** `../models/hp_opt_results.pkl` — best estimators, best params, CV results and reference metrics. Interpretation of these results relative to the AR baseline and the Scientific Validation conclusions is covered in the main ML notebook.

### Custom Scorer & TimeSeriesSplit

`neg_storm_rmse` returns negative Storm RMSE — RandomizedSearchCV maximises score, so higher = lower Storm RMSE = better model. `TimeSeriesSplit` ensures chronological order is preserved across folds. Val_Main and Val_Storm are never used during optimisation.

In [10]:
storm_scorer = make_scorer(neg_storm_rmse, greater_is_better=True)
tscv         = TimeSeriesSplit(n_splits=5, gap = H_STAR)

### XGBoost RandomizedSearchCV

50 random combinations × 5 folds = 250 fits. Results logged to MLflow.

Defines `check_degenerate_config()`, used after both searches below to flag a specific hyperparameter combination (low `learning_rate` with high `n_estimators`) previously documented in `xgboost_dst_v2.py` as a cause of prediction collapse under inverse-frequency storm weighting, and to inspect the resulting prediction distribution if that combination is selected.

In [11]:
# ── Sanity check: degenerate low-learning-rate / high-n_estimators regime ──
#
# XGBoostDst/LightGBMDst apply
# the same inverse-frequency weighting, so if RandomizedSearchCV selects that same
# combination as 'best', the resulting model is checked directly rather than trusted
# on CV score alone. Logs a tag to MLflow so the warning is visible outside this notebook.

def check_degenerate_config(search, feature_set, mask, label, mlflow_run=None):
    """
    Checks whether RandomizedSearchCV selected the documented degenerate
    (low learning_rate + high n_estimators) configuration, and if so, checks
    whether best_estimator_'s predictions on `mask` show the documented collapse.

    Parameters
    ----------
    search      : fitted RandomizedSearchCV
    feature_set : list of str — feature columns matching what `search` was fit on
    mask        : boolean Series — evaluation segment (e.g. val_main_mask)
    label       : str — 'XGBoost' or 'LightGBM', for print/tag labelling
    mlflow_run  : active mlflow run, or None to skip tagging

    Returns
    -------
    dict with keys: is_degenerate_params, frac_negative (or None if not checked)
    """
    lr    = search.best_params_.get('model__learning_rate')
    n_est = search.best_params_.get('model__n_estimators')
    print(f"[{label}] best_params_: learning_rate={lr}, n_estimators={n_est}")

    is_degenerate_params = lr is not None and lr <= 0.005 and n_est is not None and n_est >= 500
    result = {'is_degenerate_params': is_degenerate_params, 'frac_negative': None}

    if not is_degenerate_params:
        print(f"[{label}] Configuration does not match the documented degenerate regime — no further check needed.")
        return result

    print(f"[{label}]  Matches the documented degenerate regime — checking prediction distribution.")
    preds = search.best_estimator_.predict(feat_data.loc[mask, feature_set])
    frac_negative = (preds < 0).mean()
    result['frac_negative'] = frac_negative

    print(f"[{label}]   Fraction of negative predictions: {frac_negative:.3f}")
    print(f"[{label}]   Prediction range: [{preds.min():.2f}, {preds.max():.2f}] nT")

    if frac_negative > 0.95:
        print(f"[{label}]   Over 95% negative — consistent with the documented collapse. Review before accepting.")
    else:
        print(f"[{label}]   Prediction distribution does not show the documented collapse pattern.")

    if mlflow_run is not None:
        mlflow.set_tag(f'{label.lower()}_degenerate_config_flagged', is_degenerate_params)
        if frac_negative is not None:
            mlflow.set_tag(f'{label.lower()}_frac_negative_preds', round(float(frac_negative), 3))

    return result


In [12]:
# ── XGBoost RandomizedSearchCV ────────────────────────────────────────────
#
# 50 random combinations × 5 folds = 250 fits
# Val_Main and Val_Storm are NOT used — scoring on CV folds only.

param_dist_xgb = {
    'model__n_estimators'    : [300, 500, 800, 1000],
    'model__max_depth'       : [3, 5, 7],
    'model__learning_rate'   : [0.005, 0.01, 0.05],
    'model__subsample'       : [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
}

xgb_search = RandomizedSearchCV(
    estimator           = Pipeline([('model', XGBoostDst())]),
    param_distributions = param_dist_xgb,
    n_iter              = 50,
    cv                  = tscv,
    scoring             = storm_scorer,
    n_jobs              = 1,
    verbose             = 1,
    random_state        = 42,
    refit               = True,
)

t0 = time.time()
xgb_search.fit(X_train_full, y_train_full)
print(f'\nXGBoost time     : {(time.time()-t0)/60:.1f} min')
print(f'XGBoost params   : {xgb_search.best_params_}')
print(f'XGBoost CV score : {xgb_search.best_score_:.4f}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits

XGBoost time     : 45.4 min
XGBoost params   : {'model__subsample': 0.8, 'model__n_estimators': 500, 'model__max_depth': 5, 'model__learning_rate': 0.005, 'model__colsample_bytree': 0.7}
XGBoost CV score : -34.3935


Runs the check defined above against the completed XGBoost search.

In [13]:
xgb_check = check_degenerate_config(
    xgb_search, MODEL_D_OMNI_HISTORY, val_main_mask, label='XGBoost'
)

[XGBoost] best_params_: learning_rate=0.005, n_estimators=500
[XGBoost]  Matches the documented degenerate regime — checking prediction distribution.
[XGBoost]   Fraction of negative predictions: 1.000
[XGBoost]   Prediction range: [-136.30, -0.67] nT
[XGBoost]   Over 95% negative — consistent with the documented collapse. Review before accepting.


If the check above flags the degenerate combination, this cell selects the next-best configuration from the same completed search's `cv_results_` (excluding any candidate matching the same combination), refits a model with it, and overwrites `xgb_search.best_estimator_` / `best_params_` so every downstream reference to these attributes reflects the substitution automatically.

In [14]:
# ── Replace degenerate config with best non-degenerate CV candidate, if needed ──
if xgb_check['is_degenerate_params']:
    cv_df = pd.DataFrame(xgb_search.cv_results_)
    is_deg = (cv_df['param_model__learning_rate'] <= 0.005) & (cv_df['param_model__n_estimators'] >= 500)
    safe_row = cv_df[~is_deg].sort_values('mean_test_score', ascending=False).iloc[0]
    safe_params = {k.replace('model__',''): v for k, v in safe_row['params'].items()}

    xgb_search.best_estimator_ = Pipeline([('model', XGBoostDst(**safe_params))])
    xgb_search.best_estimator_.fit(X_train_full, y_train_full)
    xgb_search.best_params_ = safe_row['params']

    print(f"Replaced with safe configuration: {safe_params}")
    print(f"CV score (safe): {safe_row['mean_test_score']:.4f}  vs  original: {xgb_search.best_score_ if not xgb_check['is_degenerate_params'] else 'N/A (replaced)'}")

Replaced with safe configuration: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
CV score (safe): -34.6545  vs  original: N/A (replaced)


Logs the (possibly substituted) XGBoost result to MLflow.

In [15]:
with safe_mlflow_run(run_name='xgb_randomizedsearch') as run:
    if run is not None:
        mlflow.set_tag('model_type',  'xgboost')
        mlflow.set_tag('search_type', 'RandomizedSearchCV')
        mlflow.set_tag('n_iter', 50)
        mlflow.log_params(xgb_search.best_params_)
        mlflow.log_metric('cv_storm_rmse', -xgb_search.best_score_)
        mlflow.sklearn.log_model(
            xgb_search.best_estimator_,
            name='best_pipeline',
            skops_trusted_types=[
                'src.estimators.xgboost_dst.XGBoostDst',
                'xgboost.core.Booster',
                'xgboost.sklearn.XGBRegressor',
            ]
        )
        print('MLflow run logged.')
    else:
        print('MLflow tracking disabled (ENABLE_MLFLOW=False in src/config.py) — skipping log.')


MLflow tracking disabled (ENABLE_MLFLOW=False in src/config.py) — skipping log.


### LightGBM RandomizedSearchCV

Full grid has 972 combinations — exhaustive search would take several hours. `RandomizedSearchCV` with `n_iter=50` and `random_state=42` samples 50 random combinations × 5 folds = 250 fits. Results logged to MLflow.

In [16]:
# ── LightGBM RandomizedSearchCV ──────────────────────────────────────────
#
# Val_Main and Val_Storm are NOT used — scoring on CV folds only.

param_dist_lgbm = {
    'model__n_estimators'    : [300, 500, 800, 1000],
    'model__max_depth'       : [3, 5, 7],
    'model__learning_rate'   : [0.005, 0.01, 0.05],
    'model__subsample'       : [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
    'model__num_leaves'      : [15, 31, 63],
}

lgbm_search = RandomizedSearchCV(
    estimator           = Pipeline([('model', LightGBMDst())]),
    param_distributions = param_dist_lgbm,
    n_iter              = 50,
    cv                  = tscv,
    scoring             = storm_scorer,
    n_jobs              = 1,
    verbose             = 1,
    random_state        = 42,
    refit               = True,
)

t0 = time.time()
lgbm_search.fit(X_train_full, y_train_full)
lgbm_elapsed = (time.time() - t0) / 60

print(f'\nLightGBM RandomizedSearch time : {lgbm_elapsed:.1f} min')
print(f'LightGBM best params           : {lgbm_search.best_params_}')
print(f'LightGBM best CV score         : {lgbm_search.best_score_:.4f}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits

LightGBM RandomizedSearch time : 21.2 min
LightGBM best params           : {'model__subsample': 0.7, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__max_depth': 3, 'model__learning_rate': 0.01, 'model__colsample_bytree': 1.0}
LightGBM best CV score         : -33.7804


Runs the same degenerate-configuration check against the completed LightGBM search.

In [17]:
lgbm_check = check_degenerate_config(
    lgbm_search, MODEL_D_OMNI_HISTORY, val_main_mask, label='LightGBM'
)

[LightGBM] best_params_: learning_rate=0.01, n_estimators=500
[LightGBM] Configuration does not match the documented degenerate regime — no further check needed.


In [18]:
# ── MLflow logging (skipped if ENABLE_MLFLOW=False in src/config.py) ──────
with safe_mlflow_run(run_name='lgbm_randomizedsearch') as run:
    if run is not None:
        mlflow.set_tag('model_type',  'lightgbm')
        mlflow.set_tag('search_type', 'RandomizedSearchCV')
        mlflow.set_tag('n_iter', 50)
        mlflow.log_params(lgbm_search.best_params_)
        mlflow.log_metric('cv_storm_rmse', -lgbm_search.best_score_)
        mlflow.log_metric('search_time_min', lgbm_elapsed)
        mlflow.sklearn.log_model(
            lgbm_search.best_estimator_,
            name='best_pipeline',
            skops_trusted_types=[
                'src.estimators.lightgbm_dst.LightGBMDst',
                'lightgbm.basic.Booster',
                'lightgbm.sklearn.LGBMRegressor',
                'collections.OrderedDict',
            ]
        )
        print('MLflow run logged.')
    else:
        print('MLflow tracking disabled (ENABLE_MLFLOW=False in src/config.py) — skipping log.')


MLflow tracking disabled (ENABLE_MLFLOW=False in src/config.py) — skipping log.


### Save Artifacts

Save best estimators and params for use in the main ML notebook. `hp_opt_results.pkl` contains full CV results for diagnostic plots.

Evaluates both tuned models (XGBoost reflecting the substitution above, if it occurred) on `Val_Main` and `Val_Storm`, then saves both estimators, their parameters, full CV results, and these reference metrics to `models/hp_opt_results.pkl` for use in the main ML notebook.

In [19]:
def eval_pipe(pipe, seg_mask):
    return compute_metrics(
        y_true    = feat_data.loc[seg_mask, DST_TARGET_H],
        y_pred    = pipe.predict(feat_data.loc[seg_mask, MODEL_D_OMNI_HISTORY]),
        y_train   = y_train,
        y_persist = feat_data.loc[seg_mask, 'dst'].values,
        storm_thr = STORM_THR,
        horizon   = H_STAR,
    )

ref_metrics = {}
for model_name, pipe in [('XGBoost', xgb_pipe), ('LightGBM', lgbm_pipe)]:
    ref_metrics[model_name] = {
        seg: eval_pipe(pipe, mask)
        for seg, mask in EVAL_SEGMENTS.items()
    }

xgb_ref_metrics  = ref_metrics['XGBoost']
lgbm_ref_metrics = ref_metrics['LightGBM']

joblib.dump({
    'xgb_best_estimator' : xgb_search.best_estimator_,
    'xgb_best_params'    : xgb_search.best_params_,
    'xgb_cv_results'     : xgb_search.cv_results_,
    'lgbm_best_estimator': lgbm_search.best_estimator_,
    'lgbm_best_params'   : lgbm_search.best_params_,
    'lgbm_cv_results'    : lgbm_search.cv_results_,
    'xgb_ref_metrics'    : xgb_ref_metrics,
    'lgbm_ref_metrics'   : lgbm_ref_metrics,
    'horizon'            : H_STAR,
    'target_col'         : DST_TARGET_H,
    'feature_set'        : MODEL_D_OMNI_HISTORY,
    'cv_gap'             : H_STAR,
}, '../models/hp_opt_results.pkl')

['../models/hp_opt_results.pkl']

Ad-hoc check on the substituted XGBoost configuration's prediction distribution — fraction of negative predictions and their range on `Val_Main`, for direct comparison against the check output above.

In [20]:
preds_safe = xgb_search.best_estimator_.predict(feat_data.loc[val_main_mask, MODEL_D_OMNI_HISTORY])
print((preds_safe < 0).mean(), preds_safe.min(), preds_safe.max())

0.9946899623158616 -144.52106 2.9326978


Compares the prediction distribution above against the actual target distribution on the same segment — fraction of negative values, standard deviation, and correlation between predictions and actual values.

In [21]:
y_true_h7 = feat_data.loc[val_main_mask, DST_TARGET_H]
print("Fraction of negative values in actual Dst:", (y_true_h7 < 0).mean())
print("Std of actual Dst:", y_true_h7.std())
print("Std of 'safe' predictions:", preds_safe.std())
import numpy as np
mask_f = np.isfinite(y_true_h7.values) & np.isfinite(preds_safe)
print("Correlation(prediction, actual):", np.corrcoef(y_true_h7.values[mask_f], preds_safe[mask_f])[0,1])

Fraction of negative values in actual Dst: 0.7244299798256633
Std of actual Dst: 15.58000021037894
Std of 'safe' predictions: 15.922235
Correlation(prediction, actual): 0.698621951806266
